# 📖 Notebook 4: Fraud Detection Basics

Payment fraud costs businesses billions of dollars every year.  
Even a simple payment system needs basic fraud detection to flag suspicious activity before money leaves the account.

In this notebook we build a **rule-based fraud detection engine** — the same approach many real systems start with before adding machine learning.

## Learning Objectives

By the end of this notebook you'll understand:
- Common fraud patterns (velocity, amount anomalies, card testing)
- How to implement simple rule-based checks
- How Redis enables real-time velocity tracking with sliding windows
- How to record fraud signals in the database for auditing

## 🛠️ Setup

```bash
cd 06-system-designs/payment-system
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import redis
import json
import uuid
import time

DB_CONFIG = {
    "host": "localhost", "port": 5432,
    "database": "payment_demo", "user": "demo", "password": "demo"
}
REDIS_CONFIG = {"host": "localhost", "port": 6379, "decode_responses": True}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

conn = get_db(); r = get_redis()
print("✅ Postgres connected"); print("✅ Redis connected")
conn.close()

---
## 1. Common Fraud Patterns

Before we write code, let's understand what we're looking for:

| Pattern | What It Looks Like | Why It's Suspicious |
|---------|-------------------|--------------------|
| **Velocity** | 10+ charges from the same card in 5 minutes | Stolen card being drained quickly |
| **Large Amount** | A single charge for $10,000+ | Fraudsters go big before the card is blocked |
| **Card Testing** | Many tiny charges ($0.50–$1.00) in rapid succession | Testing if a stolen card number is valid |
| **New Card + Big Charge** | First-ever transaction on a card is very large | Legitimate customers usually start small |

We'll implement the first three as simple Python rules.

---
## 2. Rule 1: Velocity Check (Using Redis Sliding Window)

**Goal**: Flag a card if it's used more than N times in a time window.

We use a Redis **sorted set** as a sliding window:  
- Each charge adds a timestamp to the set.
- Before each charge, we remove timestamps older than the window.
- If the remaining count exceeds the threshold, we flag it.

This is O(1) per check and works in real-time — no database queries needed.

In [ ]:
VELOCITY_WINDOW_SECONDS = 300  # 5 minutes
VELOCITY_MAX_CHARGES = 5       # max charges per card in the window

def check_velocity(card_last_four, r=None):
    """
    Check if a card has been charged too many times in the sliding window.
    Uses a Redis sorted set where the score is the Unix timestamp.
    """
    if r is None:
        r = get_redis()

    key = f"fraud:velocity:{card_last_four}"
    now = time.time()
    window_start = now - VELOCITY_WINDOW_SECONDS

    # Remove timestamps outside the window
    r.zremrangebyscore(key, 0, window_start)

    # Count how many charges are in the current window
    count = r.zcard(key)

    # Add the current charge timestamp
    r.zadd(key, {f"{now}-{uuid.uuid4().hex[:8]}": now})
    r.expire(key, VELOCITY_WINDOW_SECONDS + 60)  # auto-cleanup

    flagged = count >= VELOCITY_MAX_CHARGES
    return {
        "signal": "velocity",
        "count_in_window": count + 1,
        "threshold": VELOCITY_MAX_CHARGES,
        "window_seconds": VELOCITY_WINDOW_SECONDS,
        "flagged": flagged
    }

# Demo: simulate rapid charges on the same card
r = get_redis()
r.delete("fraud:velocity:1111")  # clean slate

print(f"Velocity rule: max {VELOCITY_MAX_CHARGES} charges per {VELOCITY_WINDOW_SECONDS}s window\n")

for i in range(1, 8):
    result = check_velocity("1111", r)
    emoji = "🚨" if result["flagged"] else "✅"
    print(f"  Charge #{i}: {emoji} count={result['count_in_window']}  flagged={result['flagged']}")

---
## 3. Rule 2: Large Amount Check

**Goal**: Flag transactions above a certain dollar threshold.

This is the simplest rule but surprisingly effective. Most legitimate purchases are under a few hundred dollars.

In [ ]:
LARGE_AMOUNT_THRESHOLD_CENTS = 100000  # $1,000.00

def check_large_amount(amount_cents):
    """
    Flag transactions over a dollar threshold.
    In production, this threshold would vary per merchant and customer history.
    """
    flagged = amount_cents >= LARGE_AMOUNT_THRESHOLD_CENTS
    return {
        "signal": "large_amount",
        "amount_cents": amount_cents,
        "threshold_cents": LARGE_AMOUNT_THRESHOLD_CENTS,
        "flagged": flagged
    }

# Test with different amounts
test_amounts = [2500, 9999, 50000, 100000, 500000]
print(f"Large amount threshold: ${LARGE_AMOUNT_THRESHOLD_CENTS/100:.2f}\n")

for amt in test_amounts:
    result = check_large_amount(amt)
    emoji = "🚨" if result["flagged"] else "✅"
    print(f"  ${amt/100:>10.2f}  {emoji}  flagged={result['flagged']}")

---
## 4. Rule 3: Card Testing Detection

**Goal**: Detect when someone is testing stolen card numbers with many tiny charges.

Card testing is when a fraudster has a list of potentially stolen card numbers and makes small charges ($0.50–$2.00) to see which ones work. The small amount avoids triggering most fraud systems.

We detect this by counting small charges from the same **merchant** in a short window.  
Legitimate merchants rarely have bursts of many tiny charges.

In [ ]:
CARD_TEST_AMOUNT_THRESHOLD = 200    # charges under $2.00
CARD_TEST_WINDOW_SECONDS = 120      # 2-minute window
CARD_TEST_COUNT_THRESHOLD = 5       # 5+ tiny charges = suspicious

def check_card_testing(merchant_id, amount_cents, r=None):
    """
    Flag if a merchant sends too many tiny charges in a short window.
    This pattern usually means someone is testing stolen card numbers.
    """
    if r is None:
        r = get_redis()

    # Only track charges under the threshold
    if amount_cents >= CARD_TEST_AMOUNT_THRESHOLD:
        return {"signal": "card_testing", "flagged": False, "reason": "amount above threshold"}

    key = f"fraud:card_test:{merchant_id}"
    now = time.time()
    window_start = now - CARD_TEST_WINDOW_SECONDS

    r.zremrangebyscore(key, 0, window_start)
    count = r.zcard(key)
    r.zadd(key, {f"{now}-{uuid.uuid4().hex[:8]}": now})
    r.expire(key, CARD_TEST_WINDOW_SECONDS + 60)

    flagged = count >= CARD_TEST_COUNT_THRESHOLD
    return {
        "signal": "card_testing",
        "small_charge_count": count + 1,
        "threshold": CARD_TEST_COUNT_THRESHOLD,
        "flagged": flagged
    }

# Simulate card testing: many $1.00 charges from the same merchant
r = get_redis()
r.delete("fraud:card_test:merch_suspect")

print(f"Card testing rule: {CARD_TEST_COUNT_THRESHOLD}+ charges under ${CARD_TEST_AMOUNT_THRESHOLD/100:.2f} in {CARD_TEST_WINDOW_SECONDS}s\n")

for i in range(1, 9):
    result = check_card_testing("merch_suspect", 100, r)  # $1.00 each
    emoji = "🚨" if result["flagged"] else "✅"
    print(f"  Tiny charge #{i}: {emoji} count={result.get('small_charge_count', 'N/A')}  flagged={result['flagged']}")

---
## 5. Putting It All Together: The Fraud Check Pipeline

In a real system, you'd run ALL fraud rules before processing a charge.  
If any rule flags the transaction, you can either **block** it or **hold it for review**.

Let's build the complete pipeline and store results in our `fraud_signals` table.

In [ ]:
def run_fraud_checks(transaction_id, merchant_id, card_last_four, amount_cents):
    """
    Run all fraud rules against a transaction.
    Returns True if the transaction should be blocked.
    Records all signals in the fraud_signals table.
    """
    r = get_redis()
    signals = []

    # Run each rule
    signals.append(check_velocity(card_last_four, r))
    signals.append(check_large_amount(amount_cents))
    signals.append(check_card_testing(merchant_id, amount_cents, r))

    # Store signals in the database for auditing
    conn = get_db()
    cur = conn.cursor()
    for s in signals:
        cur.execute("""
            INSERT INTO fraud_signals (transaction_id, signal_name, signal_value, flagged, details)
            VALUES (%s, %s, %s, %s, %s)
        """, (
            transaction_id,
            s["signal"],
            s.get("count_in_window") or s.get("amount_cents") or s.get("small_charge_count"),
            s["flagged"],
            json.dumps(s)
        ))
    conn.commit()
    cur.close(); conn.close()

    # Determine overall verdict
    any_flagged = any(s["flagged"] for s in signals)

    print(f"\n  Fraud check results for {transaction_id}:")
    for s in signals:
        emoji = "🚨" if s["flagged"] else "✅"
        print(f"    {emoji} {s['signal']}: flagged={s['flagged']}")

    if any_flagged:
        print(f"\n  🛑 TRANSACTION BLOCKED — held for review")
    else:
        print(f"\n  ✅ All checks passed — transaction approved")

    return any_flagged

print("Fraud check pipeline ready.")

In [ ]:
# fraud_signals has a foreign key to transactions.id, so we need real
# payment_intents + transactions for the FK on fraud_signals to pass.
# A helper that creates the bare-minimum rows for a synthetic test transaction:
def ensure_test_txn(txn_id, merchant_id, card_last_four, amount_cents):
    conn = get_db(); cur = conn.cursor()
    try:
        pi_id = f"pi_fraud_{txn_id}"
        cur.execute("""
            INSERT INTO payment_intents (id, merchant_id, amount_cents, currency, description, status)
            VALUES (%s, %s, %s, 'usd', 'fraud demo', 'processing')
            ON CONFLICT DO NOTHING
        """, (pi_id, merchant_id, amount_cents))
        cur.execute("""
            INSERT INTO transactions (id, payment_intent_id, type, amount_cents, currency, status, card_last_four, card_brand)
            VALUES (%s, %s, 'charge', %s, 'usd', 'pending', %s, 'visa')
            ON CONFLICT DO NOTHING
        """, (txn_id, pi_id, amount_cents, card_last_four))
        conn.commit()
    finally:
        cur.close(); conn.close()

# Reset Redis counters for a clean demo
r = get_redis()
for key in r.keys("fraud:*"):
    r.delete(key)

# --- Test 1: Normal transaction (should pass) ---
print("=" * 50)
print("Test 1: Normal $25.00 charge")
ensure_test_txn("txn_test_001", "merch_001", "4242", 2500)
run_fraud_checks("txn_test_001", "merch_001", "4242", 2500)

# --- Test 2: Very large amount (should flag) ---
print()
print("=" * 50)
print("Test 2: Unusually large $5,000 charge")
ensure_test_txn("txn_test_002", "merch_001", "4242", 500000)
run_fraud_checks("txn_test_002", "merch_001", "4242", 500000)

# --- Test 3: Rapid velocity (should flag after threshold) ---
print()
print("=" * 50)
print("Test 3: 6 rapid charges on card '8888'")
for i in range(1, 7):
    print()
    print(f"--- Rapid charge #{i} ---")
    ensure_test_txn(f"txn_rapid_{i}", "merch_002", "8888", 5000)
    run_fraud_checks(f"txn_rapid_{i}", "merch_002", "8888", 5000)


---
## 6. Querying Fraud Signals

All fraud signals are stored in the database. This lets us:
- Audit why a transaction was blocked
- Analyze false positive rates
- Tune thresholds over time

In [ ]:
conn = get_db()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

# Show all flagged signals
cur.execute("""
    SELECT transaction_id, signal_name, signal_value, flagged, created_at
    FROM fraud_signals
    WHERE flagged = TRUE
    ORDER BY created_at DESC
""")

rows = cur.fetchall()
print(f"🚨 Flagged fraud signals: {len(rows)}\n")
print(f"{'Transaction':<20} {'Signal':<20} {'Value':>10}  {'Time'}")
print("-" * 70)
for r in rows:
    print(f"{r['transaction_id']:<20} {r['signal_name']:<20} {r['signal_value'] or 'N/A':>10}  {r['created_at']}")

# Summary: flag rate by signal type
cur.execute("""
    SELECT 
        signal_name,
        COUNT(*) AS total_checks,
        SUM(CASE WHEN flagged THEN 1 ELSE 0 END) AS flagged_count,
        ROUND(100.0 * SUM(CASE WHEN flagged THEN 1 ELSE 0 END) / COUNT(*), 1) AS flag_rate_pct
    FROM fraud_signals
    GROUP BY signal_name
    ORDER BY signal_name
""")

print(f"\n📊 Signal summary:")
print(f"{'Signal':<20} {'Total':>8} {'Flagged':>8} {'Flag Rate':>10}")
print("-" * 50)
for r in cur.fetchall():
    print(f"{r['signal_name']:<20} {r['total_checks']:>8} {r['flagged_count']:>8} {r['flag_rate_pct']:>9}%")

cur.close(); conn.close()

---
## 7. Summary

| Concept | How We Implemented It |
|---------|----------------------|
| **Velocity Check** | Redis sorted set as a sliding time window |
| **Large Amount** | Simple threshold comparison |
| **Card Testing** | Count small charges per merchant in a time window |
| **Signal Storage** | All results saved in `fraud_signals` table for auditing |

### What Real Systems Add

Our rule-based approach is a great starting point, but production systems also use:

- **Machine learning models** trained on historical fraud data
- **Device fingerprinting** (browser, OS, IP geolocation)
- **Behavioral analysis** (typing speed, mouse patterns)
- **Network graphs** (linking cards, devices, and merchants)
- **3D Secure** (extra authentication step for risky transactions)

### Key Takeaways

1. **Start with simple rules** — they catch a surprising amount of fraud.
2. **Redis is perfect for real-time counters** — sorted sets give you O(1) sliding windows.
3. **Always record signals** — you need them for auditing and tuning.
4. **False positives are costly** — blocking a legitimate customer is bad for business.
5. **Defense in depth** — no single rule catches everything; layer multiple signals.

🎉 **Congratulations!** You've completed the Payment System lab series.